# Rate-Distortion Knee-Point Analyse (Alemi et al., 2018 & Kneedle, Satopää et al., 2011)

Dieses Notebook bestimmt aus den Annealing-Trajektorien der Guarded-Varix-Läufe parameterfrei den **Knee Point** der Rate-Distortion-Kurve $(D, R)$ und vergleicht ihn direkt mit dem **Lucas-Handover** ($\beta_{\text{ho}}$) sowie dem **stabilisierten ControlVAE-PI-Punkt** ($\beta_{\text{PI}}$):

1. **Pareto-Front** aus $(D, R)$ ab dem Rekonstruktionsminimum (nach Warmup) filtern.
2. **Min-Max-Normalisierung** beider Achsen auf $[0, 1]$ zwischen $\beta \approx 0$ und Posterior Collapse.
3. **Kneedle-Kriterium (maximaler orthogonaler Abstand zur Sekante):**
   $$i^* = \arg\max_i \left(1 - \tilde{D}_i - \tilde{R}_i\right) \quad \text{(konvexe Front unterhalb der Sekante)}$$


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Raster-Pfade (einfach umschaltbar)
GRID_EPS02 = Path("/mnt/data/nickok/guarded_tcga_grid_ki006_eps02_hold100_2026-11-23/beta_guard_vae")
GRID_EPS01 = Path("/mnt/data/nickok/guarded_tcga_grid_ki006_eps01_hold100_2026-11-24/beta_guard_vae")

ACTIVE_GRID = GRID_EPS02
SIGNAL_MODE = "train_eval"   # "train_eval" (mse_train_eval_mu, R_train_eval_nats) oder "valid" (recon_loss, total_KL)
VARIANT = "base"             # z. B. "base", "drop_p0", "latent_dim29", ...
GROUPS = ["all_classes", "ontix_standard", "weak_signal_lowvar"]

print("Active grid:", ACTIVE_GRID, "exists:", ACTIVE_GRID.exists())


In [ ]:
def pareto_front_indices(distortion: np.ndarray, rate: np.ndarray) -> np.ndarray:
    """Indizes der nicht-dominierten Pareto-Front (kleineres D und kleineres R sind besser),
    sortiert nach steigender Distortion D (und damit fallender Rate R)."""
    order = np.lexsort((rate, distortion))
    front = []
    best_r = np.inf
    for idx in order:
        r = rate[idx]
        if r < best_r:
            front.append(idx)
            best_r = r
    return np.asarray(front, dtype=int)


def kneedle_rd(distortion: np.ndarray, rate: np.ndarray) -> dict:
    """Parameterfreie Kneedle-Bestimmung (Satopää et al., 2011) in reinem NumPy."""
    d = np.asarray(distortion, dtype=float)
    r = np.asarray(rate, dtype=float)
    if len(d) < 3:
        raise ValueError("Need at least 3 points for Kneedle analysis")

    front_idx = pareto_front_indices(d, r)
    df, rf = d[front_idx], r[front_idx]
    if len(front_idx) < 3:
        return {
            "front_idx": front_idx,
            "knee_pos": 0,
            "knee_idx": int(front_idx[0]),
            "d_norm": np.zeros_like(df),
            "r_norm": np.zeros_like(rf),
            "distance": np.zeros_like(df),
        }

    d_span = df.max() - df.min()
    r_span = rf.max() - rf.min()
    d_norm = (df - df.min()) / d_span if d_span > 0 else np.zeros_like(df)
    r_norm = (rf - rf.min()) / r_span if r_span > 0 else np.zeros_like(rf)

    # Konvexe, fallende RD-Front liegt unterhalb der Sekante d_norm + r_norm = 1.
    # Orthogonaler Abstand zur Sekante ist proportional zu (1 - d_norm - r_norm).
    signed_dist = 1.0 - d_norm - r_norm
    knee_pos = int(np.argmax(signed_dist))
    return {
        "front_idx": front_idx,
        "knee_pos": knee_pos,
        "knee_idx": int(front_idx[knee_pos]),
        "d_norm": d_norm,
        "r_norm": r_norm,
        "distance": signed_dist,
    }


In [ ]:
def load_run_rd(run_dir: Path, signal_mode: str = "train_eval") -> dict | None:
    """Lädt Annealing-Trajektorie, Kneedle-Punkt, Handover und PI-Endpunkt eines Laufs."""
    summary_path = run_dir / "guard_summary.json"
    history_path = run_dir / "collapse_history.jsonl"
    manifest_path = run_dir / "data_manifest.json"
    if not (summary_path.is_file() and history_path.is_file() and manifest_path.is_file()):
        return None

    summary = json.loads(summary_path.read_text())
    manifest = json.loads(manifest_path.read_text())
    records = [json.loads(line) for line in history_path.read_text().splitlines() if line.strip()]
    lh = summary.get("lucas_handover") or {}
    config_beta = float(summary.get("config_beta", 1.0))

    d_key, r_key = (
        ("mse_train_eval_mu", "R_train_eval_nats")
        if signal_mode == "train_eval"
        else ("recon_loss", "total_KL")
    )

    # Nur die reguläre Annealing-Phase (vor erstem Handover/Restart) betrachten
    anneal = [r for r in records if r.get("phase") == "annealing" and r.get("attempt", 0) == 0]
    # Falls der Lauf in der ersten Kollaps-Epoche direkt auf lookback_handover ging, gehört diese Epoche noch ans Ende des Annealings
    first_collapse = next((r for r in records if r.get("lookback_handover")), None)
    if first_collapse is not None and (not anneal or anneal[-1]["epoch"] < first_collapse["epoch"]):
        anneal.append(first_collapse)

    anneal = [r for r in anneal if r.get(d_key) is not None and r.get(r_key) is not None]
    if len(anneal) < 5:
        return None

    d_all = np.array([r[d_key] for r in anneal], dtype=float)
    r_all = np.array([r[r_key] for r in anneal], dtype=float)
    beta_all = np.array([r["applied_factor"] * config_beta for r in anneal], dtype=float)
    ep_all = np.array([r["epoch_number"] for r in anneal], dtype=int)
    alive_all = np.array([r.get("lucas_n_noncollapsed_train_eval", np.nan) for r in anneal], dtype=float)

    # RD-Trade-off beginnt ab dem Rekonstruktionsminimum (nach anfänglichem Warmup)
    start_idx = int(np.argmin(d_all))
    d_rd, r_rd = d_all[start_idx:], r_all[start_idx:]
    beta_rd, ep_rd, alive_rd = beta_all[start_idx:], ep_all[start_idx:], alive_all[start_idx:]

    if len(d_rd) < 3:
        return None
    knee = kneedle_rd(d_rd, r_rd)
    ki = knee["knee_idx"]

    # Stabilisierter PI-Endpunkt (falls erfolgreich)
    pi_point = None
    if lh.get("phase") == "done":
        attempt = lh.get("restarts", 0)
        ctrl = [r for r in records if r.get("attempt") == attempt and r.get("phase") in ("control", "done")]
        if ctrl:
            last = ctrl[-1]
            pi_point = {
                "D": last.get(d_key),
                "R": last.get(r_key),
                "beta": last.get("beta_batch_last", last["applied_factor"] * config_beta),
                "epoch": last["epoch_number"],
                "alive": last.get("lucas_n_noncollapsed_train_eval"),
            }

    # Handover-Punkt (letzter gesunder Checkpoint vor dem finalen Versuch)
    ho_ep = lh.get("handover_epoch")
    ho_row = next((r for r in anneal if r["epoch"] == ho_ep), None) if ho_ep is not None else None
    ho_point = None
    if ho_row is not None:
        ho_point = {
            "D": ho_row[d_key],
            "R": ho_row[r_key],
            "beta": ho_row["applied_factor"] * config_beta,
            "epoch": ho_row["epoch_number"],
            "alive": ho_row.get("lucas_n_noncollapsed_train_eval"),
        }

    return {
        "run_dir": run_dir,
        "group": manifest.get("group"),
        "variant": manifest.get("variant", "base"),
        "split": manifest.get("split_seed"),
        "seed": manifest.get("train_seed"),
        "phase": lh.get("phase"),
        "fail_reason": lh.get("fail_reason"),
        "restarts": lh.get("restarts"),
        "eps": lh.get("collapse_epsilon"),
        "d_rd": d_rd,
        "r_rd": r_rd,
        "beta_rd": beta_rd,
        "ep_rd": ep_rd,
        "alive_rd": alive_rd,
        "knee": knee,
        "knee_point": {
            "D": float(d_rd[ki]),
            "R": float(r_rd[ki]),
            "beta": float(beta_rd[ki]),
            "epoch": int(ep_rd[ki]),
            "alive": float(alive_rd[ki]),
            "distance": float(knee["distance"][knee["knee_pos"]]),
        },
        "ho_point": ho_point,
        "pi_point": pi_point,
    }


runs = []
for run_dir in sorted(ACTIVE_GRID.iterdir()):
    if not run_dir.is_dir():
        continue
    if f"-{VARIANT}-controlvae-" not in run_dir.name:
        continue
    res = load_run_rd(run_dir, signal_mode=SIGNAL_MODE)
    if res is not None:
        runs.append(res)

print(f"Loaded {len(runs)} runs for variant={VARIANT!r} ({SIGNAL_MODE})")


In [ ]:
# Plot 1: Rate-Distortion-Kurve, Pareto-Front, Kneedle-Punkt vs. Handover & PI (split=1, seed=1)
fig, axes = plt.subplots(2, len(GROUPS), figsize=(5.5 * len(GROUPS), 8.2), constrained_layout=True)

for col, group in enumerate(GROUPS):
    ax_rd = axes[0, col]
    ax_kn = axes[1, col]
    item = next((r for r in runs if r["group"] == group and r["split"] == 1 and r["seed"] == 1), None)
    if item is None:
        ax_rd.set_title(f"{group} (nicht gefunden)")
        continue

    d_rd, r_rd = item["d_rd"], item["r_rd"]
    kn = item["knee"]
    fidx = kn["front_idx"]
    kp = item["knee_point"]

    # Obere Zeile: Original-RD-Raum
    ax_rd.scatter(d_rd, r_rd, s=12, alpha=0.25, color="gray", label="Annealing-Epochen")
    ax_rd.plot(d_rd[fidx], r_rd[fidx], "-o", ms=3, lw=1.8, color="#1f77b4", label="Pareto-Front")
    ax_rd.plot([d_rd[fidx[0]], d_rd[fidx[-1]]], [r_rd[fidx[0]], r_rd[fidx[-1]]],
               "--", lw=1.0, color="black", alpha=0.5, label="Sekante")
    ax_rd.scatter([kp["D"]], [kp["R"]], s=180, marker="*", color="#2ca02c", zorder=6,
                  label=f"Kneedle (β={kp['beta']:.3f}, Ep.{kp['epoch']})")
    if item["ho_point"]:
        ho = item["ho_point"]
        ax_rd.scatter([ho["D"]], [ho["R"]], s=90, marker="^", color="#ff7f0e", zorder=5,
                      label=f"Handover (β={ho['beta']:.3f})")
    if item["pi_point"]:
        pi = item["pi_point"]
        ax_rd.scatter([pi["D"]], [pi["R"]], s=80, marker="s", color="#d62728", zorder=5,
                      label=f"PI-Ende (β={pi['beta']:.3f})")
    ax_rd.set_title(f"{group} ({VARIANT}, s1/t1, ε={item['eps']})")
    ax_rd.set_xlabel("Distortion D (MSE)")
    ax_rd.set_ylabel("Rate R (KL in nats)")
    ax_rd.legend(fontsize=8, loc="upper right")
    ax_rd.grid(True, alpha=0.25)

    # Untere Zeile: Normalisierte Front + Kneedle-Abstand (1 - D_norm - R_norm)
    ax_kn.plot(kn["d_norm"], kn["r_norm"], "-o", ms=3, color="#1f77b4", label="Normierte Front")
    ax_kn.plot([0, 1], [1, 0], "--", color="black", alpha=0.5, label="Sekante")
    ax_kn.plot(kn["d_norm"], kn["distance"], "-", lw=1.8, color="#2ca02c", label="Abstand 1 - D̃ - R̃")
    pos = kn["knee_pos"]
    ax_kn.axvline(kn["d_norm"][pos], color="#2ca02c", ls=":", alpha=0.8)
    ax_kn.scatter([kn["d_norm"][pos]], [kn["r_norm"][pos]], s=140, marker="*", color="#2ca02c", zorder=6)
    ax_kn.set_xlabel("Normierte Distortion D̃")
    ax_kn.set_ylabel("Normierte Rate R̃ / Abstand")
    ax_kn.set_title(f"Kneedle-Abstand (max = {kp['distance']:.3f})")
    ax_kn.legend(fontsize=8)
    ax_kn.grid(True, alpha=0.25)

plt.show()


In [ ]:
# Übersichtstabelle über alle Splits & Seeds: Kneedle vs. Handover vs. stabilisiertes PI
table_rows = []
for r in runs:
    kp = r["knee_point"]
    ho = r["ho_point"] or {}
    pi = r["pi_point"] or {}
    table_rows.append({
        "group": r["group"],
        "split": r["split"],
        "seed": r["seed"],
        "phase": r["phase"],
        "restarts": r["restarts"],
        "ep_knee": kp["epoch"],
        "beta_knee": kp["beta"],
        "R_knee": kp["R"],
        "D_knee": kp["D"],
        "alive_knee": kp["alive"],
        "beta_ho": ho.get("beta"),
        "beta_pi": pi.get("beta"),
        "R_pi": pi.get("R"),
        "D_pi": pi.get("D"),
        "alive_pi": pi.get("alive"),
        "beta_ratio_knee_over_pi": (kp["beta"] / pi["beta"]) if pi.get("beta") else np.nan,
    })

df = pd.DataFrame(table_rows)
display(df.groupby("group")[["beta_knee", "beta_ho", "beta_pi", "R_knee", "R_pi", "D_knee", "D_pi", "alive_knee", "alive_pi"]].agg(["mean", "std"]).round(4))
df.round(4)


In [ ]:
# Plot 2: Vergleich beta_knee vs. beta_handover vs. beta_pi über alle Seeds
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), constrained_layout=True)

ok_df = df.dropna(subset=["beta_pi"])
for i, group in enumerate(GROUPS):
    sub = ok_df[ok_df["group"] == group]
    axes[0].scatter([i - 0.15] * len(sub), sub["beta_knee"], color="#2ca02c", s=55, label="β_knee" if i == 0 else None)
    axes[0].scatter([i] * len(sub), sub["beta_ho"], color="#ff7f0e", s=55, marker="^", label="β_handover" if i == 0 else None)
    axes[0].scatter([i + 0.15] * len(sub), sub["beta_pi"], color="#d62728", s=55, marker="s", label="β_PI (stabil)" if i == 0 else None)

axes[0].set_xticks(range(len(GROUPS)))
axes[0].set_xticklabels(GROUPS, rotation=15)
axes[0].set_ylabel("Effektives β")
axes[0].set_title(f"β_knee vs. β_handover vs. β_PI ({VARIANT})")
axes[0].legend()
axes[0].grid(True, alpha=0.25)

for group, color in zip(GROUPS, ["#1f77b4", "#ff7f0e", "#2ca02c"]):
    sub = ok_df[ok_df["group"] == group]
    axes[1].scatter(sub["beta_knee"], sub["beta_pi"], s=65, color=color, label=group)
lim = max(ok_df["beta_knee"].max(), ok_df["beta_pi"].max()) * 1.1 if len(ok_df) else 1.0
axes[1].plot([0, lim], [0, lim], "--", color="gray", alpha=0.6, label="β_PI = β_knee")
axes[1].set_xlabel("β_knee (Kneedle auf Annealing-Front)")
axes[1].set_ylabel("β_PI (stabilisierter Endpunkt)")
axes[1].set_title("Korrelation: Kneedle vs. ControlVAE-PI")
axes[1].legend()
axes[1].grid(True, alpha=0.25)

plt.show()


---
## Teil 2: Kneedle auf Downstream-AUROC (Information-Bottleneck-Knick)

Hier wenden wir genau dasselbe Kneedle-Prinzip auf die **Downstream-Klassifikation** an:
- **x-Achse:** Klassifikationsfehler $E = 1 - \text{AUROC}$ (kleiner ist besser)
- **y-Achse:** Regularisierungsstärke bzw. negative Kompression ($-\log_{10}(\beta)$, kleiner = stärkere Kompression)
- **Kneedle-Punkt:** Maximal starke Kompression ($\beta$ so groß wie möglich), bevor die Downstream-AUROC spürbar einbricht!

1. **Dichte Annealing-Kurven** aus `results/class_timeline/class_auc_aligned.csv` (Epoche 100 bis Kollaps) für alle 3 TCGA-Gruppen.
2. **Vergleich mit den aktuellen Grid-Läufen** (`auroc.csv` am Handover- und PI-Endpunkt).


In [ ]:
# 2a) Kneedle auf den dichten AUROC-Annealing-Kurven (class_auc_aligned.csv)
TIMELINE_CSV = Path("/home/nickok/Projects/beta_guard_vae_local/results/class_timeline/class_auc_aligned.csv")
REPR = "mu"   # "mu" oder "sampled_z"

if TIMELINE_CSV.is_file():
    tl = pd.read_csv(TIMELINE_CSV)
    tl = tl[tl["representation"] == REPR]
    # Macro-AUROC pro (group, seed, epoch_number, beta)
    macro = (
        tl.groupby(["group", "seed", "epoch_number", "beta"], as_index=False)["auc"]
        .mean()
        .rename(columns={"auc": "macro_auc"})
        .sort_values(["group", "seed", "beta"])
    )

    auroc_knees = []
    fig, axes = plt.subplots(1, len(GROUPS), figsize=(5.5 * len(GROUPS), 4.2), constrained_layout=True)
    for ax, group in zip(axes, GROUPS):
        sub_g = macro[macro["group"] == group]
        if sub_g.empty:
            ax.set_title(f"{group} (keine Daten)")
            continue
        for seed, sub in sub_g.groupby("seed"):
            sub = sub.sort_values("beta").reset_index(drop=True)
            err = 1.0 - sub["macro_auc"].to_numpy()
            # Kompression maximieren == (-log10(beta)) minimieren (monoton fallend mit steigendem beta)
            neg_log_beta = -np.log10(np.clip(sub["beta"].to_numpy(), 1e-12, None))
            # Start ab bester AUROC (kleinstem Fehler)
            s_idx = int(np.argmin(err))
            if len(err) - s_idx < 3:
                continue
            kn = kneedle_rd(err[s_idx:], neg_log_beta[s_idx:])
            ki = s_idx + kn["knee_idx"]
            auroc_knees.append({
                "group": group,
                "seed": int(seed),
                "ep_knee_auc": int(sub.loc[ki, "epoch_number"]),
                "beta_knee_auc": float(sub.loc[ki, "beta"]),
                "macro_auc_knee": float(sub.loc[ki, "macro_auc"]),
                "macro_auc_max": float(sub["macro_auc"].max()),
                "distance": float(kn["distance"][kn["knee_pos"]]),
            })
            if seed == 1:
                ax.plot(sub["beta"], sub["macro_auc"], "-o", ms=3, lw=1.6, color="#1f77b4", label=f"Macro-AUROC ({REPR}, s1)")
                ax.scatter([sub.loc[ki, "beta"]], [sub.loc[ki, "macro_auc"]], s=180, marker="*", color="#2ca02c", zorder=6,
                           label=f"Kneedle (β={sub.loc[ki, 'beta']:.4f}, AUC={sub.loc[ki, 'macro_auc']:.3f})")
                # Vergleich mit beta_knee (RD) und beta_pi aus Teil 1
                sub_rd = df[df["group"] == group]
                if not sub_rd.empty:
                    ax.axvline(sub_rd["beta_knee"].mean(), color="#2ca02c", ls=":", alpha=0.7, label=f"Ø β_knee(RD)={sub_rd['beta_knee'].mean():.4f}")
                    if sub_rd["beta_pi"].notna().any():
                        ax.axvline(sub_rd["beta_pi"].mean(), color="#d62728", ls="--", alpha=0.7, label=f"Ø β_PI={sub_rd['beta_pi'].mean():.4f}")
        ax.set_xscale("log")
        ax.set_xlabel("β (log-Skala)")
        ax.set_ylabel("Valid Macro-AUROC")
        ax.set_title(f"{group} – Downstream-AUROC vs. β")
        ax.set_ylim(0.45, 1.02)
        ax.legend(fontsize=7.5, loc="lower left")
        ax.grid(True, alpha=0.25)
    plt.show()

    df_auc_knee = pd.DataFrame(auroc_knees)
    display(df_auc_knee.groupby("group")[["beta_knee_auc", "macro_auc_knee", "macro_auc_max"]].agg(["mean", "std"]).round(4))
else:
    print("Nicht gefunden:", TIMELINE_CSV)


In [ ]:
# 2b) Downstream-AUROC der aktuellen Grid-Läufe (auroc.csv am Handover vs. stabilisiertem PI-Ende)
grid_auc_rows = []
for r in runs:
    auc_path = r["run_dir"] / "auroc.csv"
    if not auc_path.is_file():
        continue
    a = pd.read_csv(auc_path)
    for rep in ("mu", "sampled_z"):
        sub = a[a["representation"] == rep]
        if sub.empty:
            continue
        ho_rows = sub[sub["path"].str.contains("_handover_")]
        done_rows = sub[sub["path"].str.contains("_done_")]
        grid_auc_rows.append({
            "group": r["group"],
            "split": r["split"],
            "seed": r["seed"],
            "phase": r["phase"],
            "representation": rep,
            "auroc_handover": float(ho_rows["auroc_ovo"].iloc[-1]) if not ho_rows.empty else np.nan,
            "auroc_pi_done": float(done_rows["auroc_ovo"].mean()) if not done_rows.empty else np.nan,
        })

df_grid_auc = pd.DataFrame(grid_auc_rows)
if not df_grid_auc.empty:
    display(df_grid_auc.groupby(["group", "representation"])[["auroc_handover", "auroc_pi_done"]].agg(["mean", "std"]).round(4))
